In [1]:

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/gdrive")

# Define project directory
PROJECT_DIR = Path(
    "/content/gdrive/MyDrive/ChampionsAthleticMap"
)

# Define folders for raw and processed data
RAW_DIR = PROJECT_DIR / "data" / "raw" / "nutrition"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Create folders if they don't exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Verify setup
print("Project exists:", PROJECT_DIR.exists())
print("Raw nutrition folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


Mounted at /content/gdrive
Project exists: True
Raw nutrition folder: /content/gdrive/MyDrive/ChampionsAthleticMap/data/raw/nutrition
Processed folder: /content/gdrive/MyDrive/ChampionsAthleticMap/data/processed


In [2]:

import requests
import zipfile
from pathlib import Path

# Official USDA Foundation Foods CSV download
url = (
    "https://fdc.nal.usda.gov/fdc-datasets/"
    "FoodData_Central_foundation_food_csv_2025-04-24.zip"
)

zip_path = RAW_DIR / "usda_foundation_foods.zip"

# Download the dataset
response = requests.get(url, timeout=120)
response.raise_for_status()

zip_path.write_bytes(response.content)

print("Downloaded file:", zip_path.name)
print("File size (MB):", round(zip_path.stat().st_size / 1024**2, 2))

# Inspect ZIP contents without extracting yet
with zipfile.ZipFile(zip_path, "r") as z:
    filenames = z.namelist()

print("\nFiles inside ZIP:")
for name in filenames[:15]:
    print(name)


Downloaded file: usda_foundation_foods.zip
File size (MB): 3.27

Files inside ZIP:
FoodData_Central_foundation_food_csv_2025-04-24/
FoodData_Central_foundation_food_csv_2025-04-24/food_update_log_entry.csv
FoodData_Central_foundation_food_csv_2025-04-24/agricultural_samples.csv
FoodData_Central_foundation_food_csv_2025-04-24/Download API Field Descriptions.xlsx
FoodData_Central_foundation_food_csv_2025-04-24/input_food.csv
FoodData_Central_foundation_food_csv_2025-04-24/food_attribute_type.csv
FoodData_Central_foundation_food_csv_2025-04-24/lab_method_code.csv
FoodData_Central_foundation_food_csv_2025-04-24/measure_unit.csv
FoodData_Central_foundation_food_csv_2025-04-24/food_nutrient_conversion_factor.csv
FoodData_Central_foundation_food_csv_2025-04-24/nutrient.csv
FoodData_Central_foundation_food_csv_2025-04-24/sub_sample_food.csv
FoodData_Central_foundation_food_csv_2025-04-24/market_acquisition.csv
FoodData_Central_foundation_food_csv_2025-04-24/food_attribute.csv
FoodData_Central_

In [3]:

import pandas as pd
import zipfile

# Extract the downloaded ZIP file
extract_dir = RAW_DIR/"usda_foundation_extracted"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# Locate the three important CSV files
food_path = next(extract_dir.rglob("food.csv"))
nutrient_path = next(extract_dir.rglob("nutrient.csv"))
food_nutrient_path = next(extract_dir.rglob("food_nutrient.csv"))

# Load the tables
food_df = pd.read_csv(food_path, low_memory=False)
nutrient_df = pd.read_csv(nutrient_path, low_memory=False)
food_nutrient_df = pd.read_csv(
    food_nutrient_path, low_memory=False
)

# Inspect their dimensions and columns
for name, df in [
    ("Food", food_df),
    ("Nutrient", nutrient_df),
    ("Food Nutrient", food_nutrient_df)
]:
    print(f"\n{name} table")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())



Food table
Shape: (74175, 5)
Columns: ['fdc_id', 'data_type', 'description', 'food_category_id', 'publication_date']

Nutrient table
Shape: (477, 5)
Columns: ['id', 'name', 'unit_name', 'nutrient_nbr', 'rank']

Food Nutrient table
Shape: (155243, 11)
Columns: ['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id', 'min', 'max', 'median', 'footnote', 'min_year_acquired']


In [4]:

# Check the different types of food records
print("Food data types:")
print(food_df["data_type"].value_counts())

# Look at the first 10 food records
print("\nSample food records:")
display(
    food_df[
        ["fdc_id", "data_type", "description"]
    ].head(10)
)

# Find nutrients relevant to our project
print("\nRelevant nutrient definitions:")

keywords = (
    "protein|carbohydrate|total lipid|"
    "energy|fiber"
)

relevant_nutrients = nutrient_df[
    nutrient_df["name"].str.contains(
        keywords, case=False, na=False
    )
]

display(
    relevant_nutrients[
        ["id", "name", "unit_name", "nutrient_nbr"]
    ]
)

Food data types:
data_type
sub_sample_food             62022
market_acquisition           7215
sample_food                  3717
agricultural_acquisition      810
foundation_food               411
Name: count, dtype: int64

Sample food records:


,fdc_id,data_type,description
0,319874,sample_food,"HUMMUS, SABRA CLASSIC"
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC"
2,319876,market_acquisition,"HUMMUS, SABRA CLASSIC"
3,319877,sub_sample_food,Hummus
4,319878,sub_sample_food,Hummus
5,319879,sample_food,"HUMMUS, SABRA CLASSIC"
6,319880,market_acquisition,"HUMMUS, SABRA CLASSIC"
7,319881,market_acquisition,"HUMMUS, SABRA CLASSIC"
8,319882,sub_sample_food,Hummus
9,319883,sub_sample_food,Hummus



Relevant nutrient definitions:


,id,name,unit_name,nutrient_nbr
0,2047,Energy (Atwater General Factors),KCAL,957.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0
4,1003,Protein,G,203.0
5,1004,Total lipid (fat),G,204.0
6,1005,"Carbohydrate, by difference",G,205.0
7,1006,"Fiber, crude (DO NOT USE - Archived)",G,206.0
9,1008,Energy,KCAL,208.0
51,1050,"Carbohydrate, by summation",G,205.2
54,1053,Adjusted Protein,G,257.0
63,1062,Energy,kJ,268.0


In [5]:

# Filter to Foundation Foods only
foundation_df = food_df[
    food_df["data_type"] == "foundation_food"
].copy()

# Verify the number of foods
print("Total Foundation Foods:", len(foundation_df))

# Check for duplicate food identifiers
print(
    "Unique food IDs:",
    foundation_df["fdc_id"].nunique()
)

# Display the first 20 food descriptions
display(
    foundation_df[
        ["fdc_id", "description", "publication_date"]
    ].head(20)
)

# Search for foods relevant to basketball nutrition
keywords = (
    "chicken|egg|milk|yogurt|rice|"
    "oat|banana|apple|potato|beans"
)

candidate_foods = foundation_df[
    foundation_df["description"].str.contains(
        keywords, case=False, na=False
    )
]

print("\nPotential athlete nutrition foods:")
print("Matching records:", len(candidate_foods))

display(
    candidate_foods[
        ["fdc_id", "description"]
    ].head(30)
)


Total Foundation Foods: 411
Unique food IDs: 411


,fdc_id,description,publication_date
651,321358,"Hummus, commercial",2019-04-01
652,321359,"Milk, reduced fat, fluid, 2% milkfat, with add...",2019-04-01
653,321360,"Tomatoes, grape, raw",2019-04-01
798,321505,"Salt, table, iodized",2019-04-01
904,321611,"Beans, snap, green, canned, regular pack, drai...",2019-04-01
1193,321900,"Broccoli, raw",2019-04-01
1521,322228,"Milk, lowfat, fluid, 1% milkfat, with added vi...",2019-04-01
1852,322559,"Milk, nonfat, fluid, with added vitamin A and ...",2019-04-01
2185,322892,"Milk, whole, 3.25% milkfat, with added vitamin D",2019-04-01
2414,323121,"Frankfurter, beef, unheated",2019-04-01



Potential athlete nutrition foods:
Matching records: 126


,fdc_id,description
652,321359,"Milk, reduced fat, fluid, 2% milkfat, with add..."
904,321611,"Beans, snap, green, canned, regular pack, drai..."
1521,322228,"Milk, lowfat, fluid, 1% milkfat, with added vi..."
1852,322559,"Milk, nonfat, fluid, with added vitamin A and ..."
2185,322892,"Milk, whole, 3.25% milkfat, with added vitamin D"
2737,323444,"Cheese, ricotta, whole milk"
2897,323604,"Egg, whole, raw, frozen, pasteurized"
2990,323697,"Egg, white, raw, frozen, pasteurized"
3086,323793,"Egg, white, dried"
8134,328841,"Cheese, cottage, lowfat, 2% milkfat"


In [6]:

# Keep only nutrient records for Foundation Foods
foundation_nutrients = food_nutrient_df[
    food_nutrient_df["fdc_id"].isin(
        foundation_df["fdc_id"]
    )
].copy()

# Join food descriptions with nutrient measurements
merged_df = foundation_df[
    ["fdc_id", "description"]
].merge(
    foundation_nutrients,
    on="fdc_id",
    how="inner"
)

# Join nutrient names and units
merged_df = merged_df.merge(
    nutrient_df[["id", "name", "unit_name"]],
    left_on="nutrient_id",
    right_on="id",
    how="left"
)

print("Merged dataset shape:", merged_df.shape)
print("Unique foods:", merged_df["fdc_id"].nunique())

# Inspect actual nutrient measurements
display(
    merged_df[
        ["fdc_id", "description", "name",
         "amount", "unit_name"]
    ].head(15)
)


Merged dataset shape: (19266, 15)
Unique foods: 411


,fdc_id,description,name,amount,unit_name
0,321358,"Hummus, commercial",SFA 10:0,0.000,G
1,321358,"Hummus, commercial",SFA 16:0,1.410,G
2,321358,"Hummus, commercial",PUFA 20:5 n-3 (EPA),0.000,G
3,321358,"Hummus, commercial","Tocopherol, delta",1.300,MG
4,321358,"Hummus, commercial","Magnesium, Mg",71.100,MG
5,321358,"Hummus, commercial",Galactose,0.000,G
6,321358,"Hummus, commercial","PUFA 20:2 n-6 c,c",0.005,G
7,321358,"Hummus, commercial","Fatty acids, total trans-dienoic",0.012,G
8,321358,"Hummus, commercial","PUFA 18:3 n-3 c,c,c (ALA)",0.637,G
9,321358,"Hummus, commercial","Choline, from glycerophosphocholine",1.100,MG


In [7]:

# Nutrient IDs relevant to our nutrition agent
target_nutrients = {
    1003: "Protein",
    1004: "Total Fat",
    1005: "Carbohydrate",
    1079: "Dietary Fiber",
    2047: "Energy (General Factors)",
    2048: "Energy (Specific Factors)"
}

# Filter the merged dataset
selected_nutrients_df = merged_df[
    merged_df["nutrient_id"].isin(target_nutrients)
].copy()

# Count how many foods have each nutrient
nutrient_coverage = (
    selected_nutrients_df
    .groupby("nutrient_id")["fdc_id"]
    .nunique()
    .reset_index(name="foods_with_data")
)

# Add readable nutrient names
nutrient_coverage["nutrient_name"] = (
    nutrient_coverage["nutrient_id"]
    .map(target_nutrients)
)

# Calculate percentage coverage
nutrient_coverage["coverage_pct"] = (
    nutrient_coverage["foods_with_data"]
    / foundation_df["fdc_id"].nunique()
    * 100
).round(1)

display(
    nutrient_coverage[
        ["nutrient_id", "nutrient_name",
         "foods_with_data", "coverage_pct"]
    ]
)


,nutrient_id,nutrient_name,foods_with_data,coverage_pct
0,1003,Protein,399,97.1
1,1004,Total Fat,388,94.4
2,1005,Carbohydrate,352,85.6
3,1079,Dietary Fiber,207,50.4
4,2047,Energy (General Factors),322,78.3
5,2048,Energy (Specific Factors),312,75.9


In [8]:

# Extract energy measurements
energy_df = merged_df[
    merged_df["nutrient_id"].isin([2047, 2048])
].copy()

# Check for duplicate food/nutrient combinations
duplicates = energy_df.duplicated(
    subset=["fdc_id", "nutrient_id"]
).sum()

print("Duplicate food-nutrient pairs:", duplicates)

# Reshape energy measurements into columns
energy_pivot = energy_df.pivot_table(
    index="fdc_id",
    columns="nutrient_id",
    values="amount",
    aggfunc="first"
)

energy_pivot = energy_pivot.rename(
    columns={
        2047: "energy_general",
        2048: "energy_specific"
    }
)

# Check availability and overlap
print("Foods with general energy:",
      energy_pivot["energy_general"].notna().sum())

print("Foods with specific energy:",
      energy_pivot["energy_specific"].notna().sum())

print("Foods with both:",
      energy_pivot.dropna().shape[0])

print("Foods with at least one:",
      energy_pivot.shape[0])

# Compare foods with both values
both = energy_pivot.dropna().copy()

both["difference"] = (
    both["energy_specific"] -
    both["energy_general"]
)

print("\nEnergy difference summary:")
display(both["difference"].describe())


Duplicate food-nutrient pairs: 0
Foods with general energy: 322
Foods with specific energy: 312
Foods with both: 312
Foods with at least one: 322

Energy difference summary:


,difference
count,312.000000
mean,-5.687988
std,12.975932
min,-62.000000
25%,-7.703412
50%,-3.363709
75%,2.000000
max,26.000000


In [10]:
import numpy as np
# Define nutrient IDs and output column names
nutrient_columns = {
    1003: "protein_g",
    1004: "fat_g",
    1005: "carbohydrate_g",
    1079: "fiber_g",
    2047: "energy_general_kcal",
    2048: "energy_specific_kcal"
}

# Keep only our selected nutrients
nutrition_long = merged_df[
    merged_df["nutrient_id"].isin(nutrient_columns)
].copy()

# Verify there is at most one measurement per food/nutrient
assert not nutrition_long.duplicated(
    ["fdc_id", "nutrient_id"]
).any()

# Convert nutrient rows into columns
nutrition_wide = nutrition_long.pivot(
    index="fdc_id",
    columns="nutrient_id",
    values="amount"
)

nutrition_wide = nutrition_wide.rename(
    columns=nutrient_columns
).reset_index()

# Combine nutrient columns with all Foundation Foods
nutrition_df = foundation_df[
    ["fdc_id", "description", "publication_date"]
].merge(
    nutrition_wide,
    on="fdc_id",
    how="left"
)

# Select energy method, retaining its provenance
nutrition_df["calories_kcal"] = (
    nutrition_df["energy_specific_kcal"].combine_first(
        nutrition_df["energy_general_kcal"]
    )
)

nutrition_df["energy_method"] = np.select(
    [
        nutrition_df["energy_specific_kcal"].notna(),
        nutrition_df["energy_general_kcal"].notna()
    ],
    [
        "Atwater Specific (2048)",
        "Atwater General (2047)"
    ],
    default="Missing"
)

# Add dataset provenance
nutrition_df["source"] = "USDA FoodData Central"
nutrition_df["data_type"] = "Foundation Foods"

# Validate
assert len(nutrition_df) == 411
assert nutrition_df["fdc_id"].is_unique

print("Nutrition dataset shape:", nutrition_df.shape)
print("Foods with calories:",
      nutrition_df["calories_kcal"].notna().sum())

print("\nEnergy methods:")
print(nutrition_df["energy_method"].value_counts())

display(nutrition_df.head(10))


Nutrition dataset shape: (411, 13)
Foods with calories: 322

Energy methods:
energy_method
Atwater Specific (2048)    312
Missing                     89
Atwater General (2047)      10
Name: count, dtype: int64


,fdc_id,description,publication_date,protein_g,fat_g,carbohydrate_g,fiber_g,energy_general_kcal,energy_specific_kcal,calories_kcal,energy_method,source,data_type
0,321358,"Hummus, commercial",2019-04-01,7.35,17.10,14.90,5.4,243.0,229.0,229.0,Atwater Specific (2048),USDA FoodData Central,Foundation Foods
1,321359,"Milk, reduced fat, fluid, 2% milkfat, with add...",2019-04-01,3.35,1.90,4.91,NaN,NaN,NaN,NaN,Missing,USDA FoodData Central,Foundation Foods
2,321360,"Tomatoes, grape, raw",2019-04-01,0.83,0.63,5.51,2.1,31.0,27.0,27.0,Atwater Specific (2048),USDA FoodData Central,Foundation Foods
3,321505,"Salt, table, iodized",2019-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Missing,USDA FoodData Central,Foundation Foods
4,321611,"Beans, snap, green, canned, regular pack, drai...",2019-04-01,1.04,0.39,4.11,NaN,24.0,20.0,20.0,Atwater Specific (2048),USDA FoodData Central,Foundation Foods
5,321900,"Broccoli, raw",2019-04-01,2.57,0.34,6.29,2.4,NaN,NaN,NaN,Missing,USDA FoodData Central,Foundation Foods
6,322228,"Milk, lowfat, fluid, 1% milkfat, with added vi...",2019-04-01,3.38,0.95,5.19,NaN,NaN,NaN,NaN,Missing,USDA FoodData Central,Foundation Foods
7,322559,"Milk, nonfat, fluid, with added vitamin A and ...",2019-04-01,3.43,0.08,4.89,NaN,NaN,NaN,NaN,Missing,USDA FoodData Central,Foundation Foods
8,322892,"Milk, whole, 3.25% milkfat, with added vitamin D",2019-04-01,3.28,3.20,4.67,NaN,NaN,NaN,NaN,Missing,USDA FoodData Central,Foundation Foods
9,323121,"Frankfurter, beef, unheated",2019-04-01,11.70,28.00,2.89,NaN,310.0,314.0,314.0,Atwater Specific (2048),USDA FoodData Central,Foundation Foods


In [11]:

# Save the processed USDA nutrition dataset
nutrition_path = (
    PROCESSED_DIR / "usda_foundation_nutrition.csv"
)

nutrition_df.to_csv(nutrition_path, index=False)

# Create a data-quality summary
quality_report = pd.DataFrame({
    "column": nutrition_df.columns,
    "non_missing": nutrition_df.notna().sum().values,
    "missing": nutrition_df.isna().sum().values,
    "missing_pct": (
        nutrition_df.isna().mean().values * 100
    ).round(1)
})

quality_path = (
    PROCESSED_DIR / "usda_nutrition_quality_report.csv"
)

quality_report.to_csv(quality_path, index=False)

# Verify both files
print("Nutrition file saved:", nutrition_path.exists())
print("Quality report saved:", quality_path.exists())
print("Total foods:", len(nutrition_df))

display(quality_report)


Nutrition file saved: True
Quality report saved: True
Total foods: 411


,column,non_missing,missing,missing_pct
0,fdc_id,411,0,0.0
1,description,411,0,0.0
2,publication_date,411,0,0.0
3,protein_g,399,12,2.9
4,fat_g,388,23,5.6
5,carbohydrate_g,352,59,14.4
6,fiber_g,207,204,49.6
7,energy_general_kcal,322,89,21.7
8,energy_specific_kcal,312,99,24.1
9,calories_kcal,322,89,21.7


In [12]:

# Check foods with complete core nutrition information
core_columns = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrate_g"
]

complete_foods = nutrition_df.dropna(
    subset=core_columns
).copy()

print("Total Foundation Foods:", len(nutrition_df))
print("Foods with complete core nutrients:", len(complete_foods))
print("Foods requiring further review:",
      len(nutrition_df) - len(complete_foods))

# Examine foods missing at least one core nutrient
incomplete_foods = nutrition_df[
    nutrition_df[core_columns].isna().any(axis=1)
]

display(
    incomplete_foods[
        ["fdc_id", "description"] + core_columns
    ].head(15)
)


Total Foundation Foods: 411
Foods with complete core nutrients: 321
Foods requiring further review: 90


,fdc_id,description,calories_kcal,protein_g,fat_g,carbohydrate_g
1,321359,"Milk, reduced fat, fluid, 2% milkfat, with add...",NaN,3.35,1.90,4.91
3,321505,"Salt, table, iodized",NaN,NaN,NaN,NaN
5,321900,"Broccoli, raw",NaN,2.57,0.34,6.29
6,322228,"Milk, lowfat, fluid, 1% milkfat, with added vi...",NaN,3.38,0.95,5.19
7,322559,"Milk, nonfat, fluid, with added vitamin A and ...",NaN,3.43,0.08,4.89
8,322892,"Milk, whole, 3.25% milkfat, with added vitamin D",NaN,3.28,3.20,4.67
11,323444,"Cheese, ricotta, whole milk",NaN,7.81,11.00,6.86
16,324038,"Sauce, salsa, ready-to-serve",NaN,1.44,0.19,6.74
26,325658,"Sausage, Italian, pork, mild, cooked, pan-fried",NaN,18.20,26.20,2.15
28,325992,"Sausage, turkey, breakfast links, mild, raw",NaN,16.70,10.40,0.93
